# Algorithme quasi-Newton (BFGS)

**Principe** : on évite de calculer la hessienne. On construit une approximation H_k de son inverse, uniquement à partir des gradients. La direction est d_k = −H_k ∇f(x_k) et le pas α_k est choisi par recherche linéaire d'Armijo (rebroussement).

Après chaque pas, on pose s = x_{k+1} − x_k, y = ∇f(x_{k+1}) − ∇f(x_k) et ρ = 1/(yᵀs). La mise à jour BFGS est :

H_{k+1} = (I − ρ s yᵀ) H_k (I − ρ y sᵀ) + ρ s sᵀ

On ne met à jour H que si yᵀs > 0 (condition de courbure).

**Problème test** : f(x) = ½ xᵀAx − bᵀx avec A = [[6, 2], [2, 3]], b = [2, 1], x₀ = [3, −2].

**Exemple à la main (première itération, H₀ = I)** :
- g₀ = [12, −1], donc d₀ = [−12, 1]
- f(x₀) = 17. Armijo teste α = 1 (f = 281,5), α = 0,5 (f = 46,875), puis α = 0,25 (f = 6,34), qui est accepté
- x₁ = [0, −1,75] et g₁ = [−5,5 ; −6,25]
- s₀ = [−3 ; 0,25], y₀ = [−17,5 ; −5,25] (on vérifie y₀ = A s₀) et yᵀs = 51,1875, donc ρ ≈ 0,0195

In [1]:
import numpy as np

A = np.array([[6.0, 2.0], [2.0, 3.0]])
b = np.array([2.0, 1.0])
f = lambda x: 0.5 * x @ A @ x - b @ x
grad_f = lambda x: A @ x - b

def backtracking(f, x, d, g, alpha0=1.0, c=1e-4, rho=0.5):
    alpha = alpha0
    fx = f(x)
    pente = g @ d                        # derivée directionnelle (< 0)
    while f(x + alpha * d) > fx + c * alpha * pente and alpha > 1e-16:
        alpha *= rho                     # Armijo : on divise le pas par 2
    return alpha

def quasi_newton_bfgs(f, grad, x0, tol=1e-6, max_iter=1000):
    n = len(x0)
    I = np.eye(n)
    x = np.array(x0, dtype=float)
    H = np.eye(n)                        # approximation de l'inverse de la hessienne
    g = grad(x)
    xs = [x.copy()]
    normes = [np.linalg.norm(g)]
    alphas = []
    for k in range(max_iter):
        if np.linalg.norm(g) < tol:
            break
        d = -H @ g                       # direction quasi-Newton
        alpha = backtracking(f, x, d, g)
        x_new = x + alpha * d
        g_new = grad(x_new)
        s = x_new - x
        y = g_new - g
        sy = y @ s
        if sy > 1e-12:                   # condition de courbure
            rho = 1.0 / sy
            H = (I - rho * np.outer(s, y)) @ H @ (I - rho * np.outer(y, s)) \
                + rho * np.outer(s, s)
        x, g = x_new, g_new
        xs.append(x.copy())
        normes.append(np.linalg.norm(g))
        alphas.append(alpha)
    return x, xs, normes, alphas, H

x0 = [3.0, -2.0]
x_star = np.linalg.solve(A, b)
x, xs, normes, alphas, H = quasi_newton_bfgs(f, grad_f, x0)

print("x =", x, "| itérations =", len(xs) - 1)
print("erreur =", np.linalg.norm(x - x_star))
print("alpha_0 =", alphas[0])
print("x1 =", xs[1])
print("H final =\n", H)
print("A^-1 =\n", np.linalg.inv(A))

x = [0.28571429 0.14285714] | itérations = 6
erreur = 3.9129196530763676e-10
alpha_0 = 0.25
x1 = [ 0.   -1.75]
H final =
 [[ 0.21442861 -0.14292232]
 [-0.14292232  0.42860116]]
A^-1 =
 [[ 0.21428571 -0.14285714]
 [-0.14285714  0.42857143]]


**Conclusion** : BFGS converge en 6 itérations, avec une erreur de l'ordre de 4·10⁻¹⁰. L'approximation finale H est très proche de A⁻¹ : l'algorithme a reconstruit l'inverse de la hessienne sans jamais la calculer. Sa convergence est superlinéaire, entre le gradient (linéaire) et Newton (quadratique).